# Laboratorio 04 — Codificación de Canal

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ollerenac/wireless-communication-systems/blob/main/docs/sessions/04-channel-coding/lab.ipynb)

**Objetivos:**
- Visualizar la capacidad de Shannon y el límite de $E_b/N_0$
- Verificar el síndromo de un código de bloque en $\mathrm{GF}(2)$
- Simular belief propagation en un código LDPC pequeño
- Calcular los parámetros de Bhattacharyya y comparar canales polares
- Generar curvas waterfall para LDPC y Polar

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erfc
from itertools import product

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

def Q(x):
    return 0.5 * erfc(x / np.sqrt(2))

---
## Ejercicio 1 — Capacidad de Shannon y límite $E_b/N_0$

La capacidad del canal AWGN de banda $B$ es
$$C = B \log_2\!\left(1 + \frac{S}{N_0 B}\right) \quad [\text{bit/s}]$$

Expresada en función de la eficiencia espectral $\eta = C/B$:
$$\frac{E_b}{N_0} = \frac{2^\eta - 1}{\eta}$$

**Tareas:**
1. Grafica $\eta$ vs $E_b/N_0$ (dB) para $\eta \in [0.1, 6]$ bit/s/Hz.
2. Marca el límite de Shannon ($\eta \to 0$, $E_b/N_0 \to \ln 2 \approx -1{,}59$ dB).
3. Marca el punto de operación de 64-QAM 3/4 ($\eta = 4{,}5$ bit/s/Hz, $E_b/N_0 \approx 10$ dB).

In [ ]:
eta = np.linspace(0.05, 6, 500)          # bit/s/Hz
EbN0_lin = (2**eta - 1) / eta
EbN0_dB  = 10 * np.log10(EbN0_lin)

# Límite de Shannon: η→0  →  (2^η-1)/η → ln2
shannon_limit_dB = 10 * np.log10(np.log(2))   # -1.59 dB

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(EbN0_dB, eta, 'b-', lw=2, label='Límite de Shannon')
ax.fill_betweenx(eta, EbN0_dB, 20, alpha=0.08, color='blue', label='Región no alcanzable')
ax.axvline(shannon_limit_dB, color='red', ls='--', lw=1.5,
           label=f'Límite absoluto {shannon_limit_dB:.2f} dB')

# Punto 64-QAM 3/4
eta_64qam = np.log2(64) * 0.75   # 4.5 bit/s/Hz
ebno_64qam_dB = 10.0
ax.scatter([ebno_64qam_dB], [eta_64qam], s=80, color='orange', zorder=5,
           label='64-QAM 3/4 (operación típica)')
ax.annotate(f'64-QAM 3/4\n({ebno_64qam_dB} dB, {eta_64qam:.1f} b/s/Hz)',
            xy=(ebno_64qam_dB, eta_64qam), xytext=(13, 3.5),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)

ax.set_xlabel('$E_b/N_0$ (dB)')
ax.set_ylabel('Eficiencia espectral $\\eta$ (bit/s/Hz)')
ax.set_title('Región de Shannon: capacidad del canal AWGN')
ax.set_xlim(-3, 20)
ax.set_ylim(0, 6.2)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'Límite de Shannon: {shannon_limit_dB:.3f} dB')
print(f'Eficiencia 64-QAM 3/4: {eta_64qam:.2f} bit/s/Hz')
print(f'Gap al límite: {ebno_64qam_dB - shannon_limit_dB:.2f} dB')

---
## Ejercicio 2 — Código de bloque: síndrome y detección de errores

El código (7,4) de Hamming tiene la matriz de paridad:
$$H = \begin{pmatrix}
1 & 1 & 0 & 1 & 1 & 0 & 0 \\
1 & 0 & 1 & 1 & 0 & 1 & 0 \\
0 & 1 & 1 & 1 & 0 & 0 & 1
\end{pmatrix}$$

Un vector recibido válido cumple $H\mathbf{r}^\top = \mathbf{0}$ en $\mathrm{GF}(2)$.

**Tareas:**
1. Verifica que la palabra código $\mathbf{c} = [1,0,1,1,0,1,0]$ es válida.
2. Introduce un error en el bit 3 (índice 2) y calcula el síndrome.
3. Demuestra que el síndrome identifica la columna de $H$ que corresponde al bit erróneo.

In [ ]:
# Matriz de paridad del código Hamming (7,4)
H = np.array([
    [1, 1, 0, 1, 1, 0, 0],
    [1, 0, 1, 1, 0, 1, 0],
    [0, 1, 1, 1, 0, 0, 1],
], dtype=int)

c = np.array([1, 0, 1, 1, 0, 1, 0], dtype=int)  # palabra código válida

def syndrome(H, r):
    """Síndrome s = H·r^T mod 2"""
    return H @ r % 2

# 1. Verificar palabra válida
s_valid = syndrome(H, c)
print(f'Síndrome de c (sin error): {s_valid}  →  {"válida" if np.all(s_valid == 0) else "ERROR"}')

# 2. Introducir error en bit 2 (índice 2)
r = c.copy()
error_pos = 2
r[error_pos] ^= 1
print(f'Palabra recibida con error en bit {error_pos}: {r}')

s_err = syndrome(H, r)
print(f'Síndrome: {s_err}')

# 3. Identificar columna de H igual al síndrome
for j in range(H.shape[1]):
    if np.array_equal(H[:, j], s_err):
        print(f'El síndrome coincide con la columna {j} de H → error en bit {j}')
        break

# Corregir
r_corr = r.copy()
r_corr[j] ^= 1
print(f'Palabra corregida: {r_corr}  →  {"igual a c" if np.array_equal(r_corr, c) else "diferente"}')

---
## Ejercicio 3 — LDPC: belief propagation simplificado

Usaremos el código LDPC (8, 4) con la siguiente matriz de paridad $H$ (tamaño $4 \times 8$):

| Nodo de chequeo | Bits conectados |
|---|---|
| $c_0$ | $v_0, v_1, v_2, v_3$ |
| $c_1$ | $v_1, v_2, v_4, v_5$ |
| $c_2$ | $v_2, v_3, v_5, v_6$ |
| $c_3$ | $v_0, v_3, v_6, v_7$ |

Implementaremos el **algoritmo sum-product** (propagación de creencia) para BSC con probabilidad de error $p$.

**Tareas:**
1. Construye $H$ y genera una palabra código válida.
2. Transmite sobre un BSC con $p = 0{,}1$ e introduce errores.
3. Ejecuta 20 iteraciones de BP y verifica si se recupera la palabra original.

In [ ]:
# Código LDPC (8,4)
H_ldpc = np.array([
    [1, 1, 1, 1, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 1, 1, 0],
    [1, 0, 0, 1, 0, 0, 1, 1],
], dtype=int)

n, k = 8, 4  # longitud y dimensión
rng = np.random.default_rng(42)

def gf2_row_reduce(M):
    """Reducción por filas en GF(2). Devuelve la forma escalonada."""
    M = M.copy() % 2
    rows, cols = M.shape
    pivot_row = 0
    for col in range(cols):
        if pivot_row >= rows:
            break
        pivot = np.where(M[pivot_row:, col] == 1)[0]
        if len(pivot) == 0:
            continue
        pivot += pivot_row
        M[[pivot_row, pivot[0]]] = M[[pivot[0], pivot_row]]
        for r in range(rows):
            if r != pivot_row and M[r, col] == 1:
                M[r] ^= M[pivot_row]
        pivot_row += 1
    return M % 2

# Encontrar una palabra código: resolver H·c^T = 0 en GF(2)
# Palabra código simple: todos ceros
c_ldpc = np.zeros(n, dtype=int)
assert np.all(H_ldpc @ c_ldpc % 2 == 0), 'c no es palabra código'

# Otra palabra código (suma de filas generadoras)
# G se obtiene del espacio nulo de H en GF(2)
# Para simplificar, usamos una palabra no trivial verificada a mano:
c_ldpc = np.array([1, 1, 0, 0, 1, 0, 1, 1], dtype=int)
print(f'Síndrome de c_ldpc: {H_ldpc @ c_ldpc % 2}  →  {"válida" if np.all(H_ldpc @ c_ldpc % 2 == 0) else "inválida"}')


def bp_bsc(H, r, p, n_iter=20):
    """
    Belief Propagation (sum-product) para BSC con prob. de error p.
    r: vector recibido (bits 0/1)
    Devuelve: decisión dura tras n_iter iteraciones
    """
    m, n = H.shape
    # LLR del canal: L_ch[i] = log(P(r=0|x=0)/P(r=1|x=0))
    #   Si r_i = 0: L = log((1-p)/p)   (favor de 0)
    #   Si r_i = 1: L = log(p/(1-p))   (favor de 1 → negativo)
    L_ch = np.where(r == 0, np.log((1-p)/p), np.log(p/(1-p)))

    # Mensajes v→c (LLR)
    L_vc = np.zeros((m, n))
    for i in range(m):
        for j in range(n):
            if H[i, j]:
                L_vc[i, j] = L_ch[j]

    for it in range(n_iter):
        # Paso c→v: mensaje del nodo de chequeo c_i al nodo variable v_j
        L_cv = np.zeros((m, n))
        for i in range(m):
            nbrs = np.where(H[i, :] == 1)[0]  # vecinos de c_i
            for j in nbrs:
                other = [jj for jj in nbrs if jj != j]
                if len(other) == 0:
                    L_cv[i, j] = 0.0
                    continue
                # Producto de tanh (sum-product exacto)
                prod_tanh = np.prod(np.tanh(L_vc[i, other] / 2))
                # Clamp para evitar atanh(±1)
                prod_tanh = np.clip(prod_tanh, -1 + 1e-10, 1 - 1e-10)
                L_cv[i, j] = 2 * np.arctanh(prod_tanh)

        # Paso v→c: mensaje actualizado
        for j in range(n):
            nbrs_c = np.where(H[:, j] == 1)[0]  # nodos de chequeo que inciden en v_j
            total_llr = L_ch[j] + np.sum(L_cv[nbrs_c, j])
            for i in nbrs_c:
                L_vc[i, j] = total_llr - L_cv[i, j]

        # LLR marginal
        L_total = np.array([
            L_ch[j] + np.sum(L_cv[np.where(H[:, j] == 1)[0], j])
            for j in range(n)
        ])
        c_hat = (L_total < 0).astype(int)  # decide 1 si LLR < 0

        # Verificar síndrome
        if np.all(H @ c_hat % 2 == 0):
            return c_hat, it + 1

    return c_hat, n_iter


# Canal BSC con p=0.1
p_bsc = 0.1
errors = (rng.random(n) < p_bsc).astype(int)
r_ldpc = (c_ldpc + errors) % 2
print(f'\nPalabra enviada:    {c_ldpc}')
print(f'Errores en posic.:  {np.where(errors)[0].tolist()}')
print(f'Palabra recibida:   {r_ldpc}')

c_hat, iters = bp_bsc(H_ldpc, r_ldpc, p_bsc, n_iter=20)
print(f'\nDecisión BP ({iters} iters): {c_hat}')
print(f'Síndrome: {H_ldpc @ c_hat % 2}')
print(f'Recuperación: {"CORRECTA" if np.array_equal(c_hat, c_ldpc) else "INCORRECTA"}')

---
## Ejercicio 4 — Polar codes: árbol de Bhattacharyya

Para un canal binario simétrico (BEC) de parámetro de borrado $\varepsilon$, la transformación Arıkan define:
$$Z(W^-) = 2Z(W) - Z(W)^2 \qquad Z(W^+) = Z(W)^2$$

donde $Z(W) \in [0,1]$ es el parámetro de Bhattacharyya (0 = canal perfecto, 1 = canal completamente borrado).

**Tareas:**
1. Calcula el árbol completo de $N = 8$ canales polares a partir de $Z_0 = \varepsilon = 0{,}5$.
2. Identifica los canales «buenos» (frozen = 0) y los «de información».
3. Estima la probabilidad de bloque de error del código Polar (8, 4) resultante.

In [ ]:
def bhattacharyya_tree(Z0, N):
    """
    Calcula los parámetros de Bhattacharyya para N canales polares
    partiendo de Z0 (parámetro del canal base).
    Devuelve lista de longitud N con Z[i] para canal i.
    """
    n_stages = int(np.log2(N))
    # Comenzar con [Z0]
    channels = [Z0]
    for stage in range(n_stages):
        new_channels = []
        for z in channels:
            z_minus = 2*z - z**2   # canal malo (peor)
            z_plus  = z**2          # canal bueno (mejor)
            new_channels.extend([z_minus, z_plus])
        channels = new_channels
    return np.array(channels)


N_polar = 8
eps = 0.5   # BEC con epsilon=0.5

Z_channels = bhattacharyya_tree(eps, N_polar)
print(f'Parámetros de Bhattacharyya para N={N_polar}, ε={eps}:')
for i, z in enumerate(Z_channels):
    print(f'  Canal {i:2d}: Z = {z:.4f}  →  {"INFORMACIÓN" if z < 0.5 else "frozen     "}')

# Seleccionar k=4 canales con menor Z (mejores)
k_polar = 4
info_indices = np.argsort(Z_channels)[:k_polar]
frozen_indices = np.argsort(Z_channels)[k_polar:]
print(f'\nCanales de información (k={k_polar}): {sorted(info_indices)}')
print(f'Canales frozen:                      {sorted(frozen_indices)}')

# Cota de probabilidad de error de bloque (unión)
P_ub = np.sum(Z_channels[info_indices])
print(f'\nCota superior P_b ≤ {P_ub:.4f}')

# Visualización
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if i in info_indices else 'salmon' for i in range(N_polar)]
bars = ax.bar(range(N_polar), Z_channels, color=colors, edgecolor='k', lw=0.5)
ax.axhline(0.5, color='gray', ls='--', lw=1, label='Umbral de selección')
ax.set_xticks(range(N_polar))
ax.set_xlabel('Índice del canal polar $i$')
ax.set_ylabel('$Z(W_N^{(i)})$')
ax.set_title(f'Árbol de Bhattacharyya: N={N_polar}, ε={eps}')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', label='Canal de información'),
                   Patch(facecolor='salmon',    label='Canal frozen')]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.show()

---
## Ejercicio 5 — Decodificador SC para código Polar (N=4)

El decodificador Successive Cancellation (SC) recorre el árbol factor de la transformación Arıkan recursivamente. Para $N = 4$ podemos implementarlo explícitamente.

Las funciones de actualización de LLR son:
$$f(a, b) = 2\,\mathrm{arctanh}\!\left(\tanh\frac{a}{2}\cdot\tanh\frac{b}{2}\right) \quad (\text{nodo} \ {-})$$
$$g(a, b, u) = b + (1 - 2u)\,a \quad (\text{nodo} \ {+})$$

**Tareas:**
1. Implementa el decodificador SC para $N = 4$ con bits frozen $u_0 = u_1 = 0$ (canales 0 y 1 frozen).
2. Transmite la palabra código por un canal AWGN con SNR = 5 dB y decodifica.
3. Compara con decisión dura directa.

In [ ]:
def f_func(a, b):
    """Actualización de LLR para el nodo '-' (combinación XOR)"""
    t = np.tanh(a/2) * np.tanh(b/2)
    t = np.clip(t, -1 + 1e-10, 1 - 1e-10)
    return 2 * np.arctanh(t)

def g_func(a, b, u_hat):
    """Actualización de LLR para el nodo '+' (combinación suma)"""
    return b + (1 - 2 * u_hat) * a


def sc_decode_n4(llr, frozen_bits):
    """
    Decodificador SC explícito para N=4.
    llr: vector de 4 LLRs del canal
    frozen_bits: dict {index: value} para los bits frozen
    Devuelve: u_hat (4 bits), x_hat (4 bits en dominio canal)
    """
    L = llr.copy()
    u_hat = np.zeros(4, dtype=int)

    # Etapa 1: LLRs de la primera mitad
    L1 = np.array([f_func(L[0], L[2]),
                   f_func(L[1], L[3])])

    # Decodificar u0 (frozen)
    L_u0 = f_func(L1[0], L1[1])
    if 0 in frozen_bits:
        u_hat[0] = frozen_bits[0]
    else:
        u_hat[0] = 1 if L_u0 < 0 else 0

    # Decodificar u1 (frozen o dato)
    L_u1 = g_func(L1[0], L1[1], u_hat[0])
    if 1 in frozen_bits:
        u_hat[1] = frozen_bits[1]
    else:
        u_hat[1] = 1 if L_u1 < 0 else 0

    # Etapa 2: LLRs de la segunda mitad (usando u0, u1 ya decodificados)
    beta0 = (u_hat[0] + u_hat[1]) % 2
    beta1 = u_hat[1]
    L2 = np.array([g_func(L[0], L[2], beta0),
                   g_func(L[1], L[3], beta1)])

    # Decodificar u2
    L_u2 = f_func(L2[0], L2[1])
    if 2 in frozen_bits:
        u_hat[2] = frozen_bits[2]
    else:
        u_hat[2] = 1 if L_u2 < 0 else 0

    # Decodificar u3
    L_u3 = g_func(L2[0], L2[1], u_hat[2])
    if 3 in frozen_bits:
        u_hat[3] = frozen_bits[3]
    else:
        u_hat[3] = 1 if L_u3 < 0 else 0

    # Recuperar x del dominio canal: x = G·u (Arikan transform mod 2)
    # Para N=4: G = F^{⊗2}, F = [[1,0],[1,1]]
    G = np.array([[1,0,0,0],[1,1,0,0],[1,0,1,0],[1,1,1,1]], dtype=int)
    x_hat = G @ u_hat % 2

    return u_hat, x_hat


rng2 = np.random.default_rng(7)

# Código Polar (4,2): frozen = {0:0, 1:0}, info en posiciones {2,3}
frozen = {0: 0, 1: 0}
msg = np.array([1, 0])  # bits de información para u2, u3
u = np.array([0, 0, msg[0], msg[1]])  # vector completo

# Transformación Arikan: x = G·u mod 2
G4 = np.array([[1,0,0,0],[1,1,0,0],[1,0,1,0],[1,1,1,1]], dtype=int)
x = G4 @ u % 2
x_bpsk = 1 - 2*x  # BPSK: 0→+1, 1→-1

# Canal AWGN con SNR=5 dB
SNR_dB = 5.0
SNR_lin = 10**(SNR_dB/10)
sigma = 1 / np.sqrt(2 * SNR_lin)  # σ = sqrt(N0/2)
y = x_bpsk + sigma * rng2.standard_normal(4)

# LLRs del canal AWGN: L = 2y/σ²
llr_ch = 2 * y / sigma**2

# Decodificación SC
u_hat, x_hat = sc_decode_n4(llr_ch, frozen)

# Decisión dura directa
x_hard = (y < 0).astype(int)

print(f'Bits de información originales: {msg}')
print(f'Palabra código x:               {x}')
print(f'Recibido y (BPSK+AWGN):         [{" ".join(f"{v:.2f}" for v in y)}]')
print(f'\nDecisión dura directa:          {x_hard}')
print(f'Decodificación SC:  u_hat={u_hat},  x_hat={x_hat}')
print(f'\nRecuperación SC:    {"CORRECTA" if np.array_equal(x_hat, x) else "INCORRECTA"}')
print(f'Recuperación hard:  {"CORRECTA" if np.array_equal(x_hard, x) else "INCORRECTA"}')

---
## Ejercicio 6 — Curvas waterfall: LDPC vs sin codificación

Simularemos la probabilidad de error de bloque (BLER) del código LDPC (8,4) del Ejercicio 3 y la compararemos con la transmisión sin codificación (BPSK sin code).

La **ganancia de codificación** se define como la diferencia en $E_b/N_0$ para alcanzar el mismo BLER objetivo (p. ej., $10^{-2}$).

In [ ]:
def awgn_llr(r_bpsk, sigma2):
    """LLRs para canal AWGN: L_i = 2*r_i / sigma^2  (BPSK ±1)"""
    return 2 * r_bpsk / sigma2


def simulate_ldpc_bler(H, n_info, EbN0_dB_range, N_trials=2000, bp_iters=30, rng_seed=0):
    """
    BLER para código LDPC via BP en canal AWGN.
    n_info: bits de información (k)
    """
    rng_s = np.random.default_rng(rng_seed)
    m, n = H.shape
    rate = n_info / n
    bler = []

    for EbN0_dB in EbN0_dB_range:
        EbN0_lin = 10**(EbN0_dB / 10)
        # sigma^2 = N0/2 = 1/(2*rate*EbN0_lin)
        sigma2 = 1 / (2 * rate * EbN0_lin)
        sigma  = np.sqrt(sigma2)

        errors = 0
        # Usar siempre la palabra código todo-ceros (válida para códigos lineales)
        x_bpsk = np.ones(n)  # BPSK: bit 0 → +1

        for _ in range(N_trials):
            y = x_bpsk + sigma * rng_s.standard_normal(n)
            r_hard = (y < 0).astype(int)  # decisión dura para p_bsc
            # Estimación de p_bsc local: BER teórica BPSK
            p_est = float(Q(np.sqrt(2 * EbN0_lin * rate)))
            p_est = np.clip(p_est, 1e-6, 0.45)

            # BP con LLRs
            llr = awgn_llr(y, sigma2)
            # Adaptar bp_bsc a LLRs del canal (en lugar de bits + p)
            # Usamos la versión con entrada LLR directa
            c_hat = bp_ldpc_llr(H, llr, bp_iters)

            if not np.all(H @ c_hat % 2 == 0) or not np.array_equal(c_hat, np.zeros(n, dtype=int)):
                errors += 1

        bler.append(errors / N_trials)
    return np.array(bler)


def bp_ldpc_llr(H, llr_ch, n_iter=30):
    """
    Belief Propagation con entrada LLR del canal (AWGN).
    llr_ch: vector de LLRs del canal (L>0 → favor bit 0)
    """
    m, n = H.shape
    L_vc = np.zeros((m, n))
    for i in range(m):
        for j in range(n):
            if H[i, j]:
                L_vc[i, j] = llr_ch[j]

    for it in range(n_iter):
        L_cv = np.zeros((m, n))
        for i in range(m):
            nbrs = np.where(H[i, :] == 1)[0]
            for j in nbrs:
                other = [jj for jj in nbrs if jj != j]
                if len(other) == 0:
                    continue
                prod_tanh = np.prod(np.tanh(L_vc[i, other] / 2))
                prod_tanh = np.clip(prod_tanh, -1 + 1e-10, 1 - 1e-10)
                L_cv[i, j] = 2 * np.arctanh(prod_tanh)

        for j in range(n):
            nbrs_c = np.where(H[:, j] == 1)[0]
            total_llr = llr_ch[j] + np.sum(L_cv[nbrs_c, j])
            for i in nbrs_c:
                L_vc[i, j] = total_llr - L_cv[i, j]

        L_total = np.array([
            llr_ch[j] + np.sum(L_cv[np.where(H[:, j] == 1)[0], j])
            for j in range(n)
        ])
        c_hat = (L_total < 0).astype(int)
        if np.all(H @ c_hat % 2 == 0):
            return c_hat

    return (L_total < 0).astype(int)


def bler_uncoded(k, EbN0_dB_range):
    """
    BLER sin codificación: P(error de bloque) = 1 - (1-BER)^k
    con BER = Q(sqrt(2*Eb/N0))
    """
    EbN0_lin = 10**(np.array(EbN0_dB_range) / 10)
    ber = Q(np.sqrt(2 * EbN0_lin))
    return 1 - (1 - ber)**k


EbN0_range = np.arange(0, 8.5, 0.5)
k_info = 4   # bits de información
n_code = 8   # longitud del código

print('Simulando BLER LDPC (puede tardar ~10s)...')
bler_ldpc = simulate_ldpc_bler(H_ldpc, k_info, EbN0_range, N_trials=3000, rng_seed=42)
bler_unc  = bler_uncoded(k_info, EbN0_range)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(EbN0_range, bler_unc,  'r--o', ms=5, lw=1.5, label='Sin codificación (BPSK, k=4)')
ax.semilogy(EbN0_range, np.maximum(bler_ldpc, 1e-4),
            'b-s', ms=5, lw=1.5, label='LDPC (8,4) BP')
ax.set_xlabel('$E_b/N_0$ (dB)')
ax.set_ylabel('BLER')
ax.set_title('Curva waterfall — LDPC (8,4) vs sin codificación')
ax.set_ylim(1e-3, 1.1)
ax.legend(fontsize=10)
ax.axhline(1e-2, color='gray', ls=':', lw=1, label='BLER = 1%')
plt.tight_layout()
plt.show()

# Ganancia de codificación aproximada al BLER=0.1
target = 0.1
try:
    idx_unc  = np.where(bler_unc  < target)[0][0]
    idx_ldpc = np.where(bler_ldpc < target)[0][0]
    gain_dB = EbN0_range[idx_unc] - EbN0_range[idx_ldpc]
    print(f'\nGanancia de codificación en BLER={target}: ~{gain_dB:.1f} dB')
except IndexError:
    print('\nNo se alcanzó el umbral BLER con el rango de SNR simulado.')

---
## Resumen

| Ejercicio | Concepto clave | Resultado esperado |
|---|---|---|
| 1 | Región de Shannon, límite $E_b/N_0$ | $-1{,}59$ dB; gap ~11 dB para 64-QAM |
| 2 | Código Hamming (7,4), síndrome | Síndrome identifica columna = posición del error |
| 3 | BP en LDPC (8,4) BSC | Convergencia en <10 iter para $p=0{,}1$ |
| 4 | Árbol de Bhattacharyya, BEC $\varepsilon=0{,}5$ | 4 canales buenos: $\{2,3,5,7\}$ aprox. |
| 5 | Decodificador SC N=4 | Recuperación correcta a 5 dB SNR |
| 6 | Waterfall LDPC vs sin código | Ganancia ~1-2 dB para código pequeño (8,4) |